In [1]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2 — Imports
# ============================================================

import json
import math
from pathlib import Path

import pandas as pd

In [3]:
# ============================================================
# CELL 3 — Evaluation Paths
# ============================================================

EVAL_DIR = Path(
    "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/"
    "eval/retrieval_only/retrieval_local"
)

BENCHMARK_DIR = EVAL_DIR / "benchmarks"

RUN_DIR = (
    EVAL_DIR /
    "runs" /
    "local_retrieval_v1"
)

SCORES_DIR = EVAL_DIR / "scores"

REPORTS_DIR = EVAL_DIR / "reports"

SCORES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Evaluation directory:")
print(EVAL_DIR)

print("\nRun directory:")
print(RUN_DIR)

print("\nScores directory:")
print(SCORES_DIR)

print("\nReports directory:")
print(REPORTS_DIR)

Evaluation directory:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local

Run directory:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/runs/local_retrieval_v1

Scores directory:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/scores

Reports directory:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/reports


In [4]:
# ============================================================
# CELL 4 — Load Benchmark
# ============================================================

benchmark_files = list(
    BENCHMARK_DIR.glob("*.json")
)

preferred = [
    p for p in benchmark_files
    if "aria_local_benchmark_v2" in p.name.lower()
]

if not preferred:
    raise FileNotFoundError(
        "Could not find aria_local_benchmark_v2."
    )

BENCHMARK_PATH = preferred[0]

with open(
    BENCHMARK_PATH,
    "r",
    encoding="utf-8"
) as f:

    benchmark = json.load(f)

queries = benchmark["queries"]

print(
    f"Loaded {len(queries)} benchmark queries."
)

Loaded 10 benchmark queries.


In [5]:
# ============================================================
# CELL 5 — Load Raw Retrieval Runs
# ============================================================

run_files = sorted(
    RUN_DIR.glob("Q*.json")
)

if not run_files:
    raise FileNotFoundError(
        f"No query result files found in {RUN_DIR}"
    )

raw_runs = {}

for path in run_files:

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        record = json.load(f)

    raw_runs[record["query_id"]] = record

print(
    f"Loaded {len(raw_runs)} retrieval runs."
)

print("\nQueries found:")

for query_id in sorted(raw_runs):
    print(" -", query_id)

Loaded 10 retrieval runs.

Queries found:
 - Q1
 - Q10
 - Q2
 - Q3
 - Q4
 - Q5
 - Q6
 - Q7
 - Q8
 - Q9


In [6]:
# ============================================================
# CELL 6 — Inspect One Retrieval Result
# ============================================================

first_query_id = queries[0]["id"]

example_run = raw_runs[first_query_id]

print(
    "Query:",
    first_query_id
)

print("\nRetrieval result type:")
print(
    type(
        example_run["retrieval"]
    )
)

print("\nNumber of retrieved items:")
print(
    len(
        example_run["retrieval"]
    )
)

print("\nFirst retrieved item:")

print(
    json.dumps(
        example_run["retrieval"][0],
        indent=2,
        ensure_ascii=False,
        default=str
    )
)

Query: Q1

Retrieval result type:
<class 'list'>

Number of retrieved items:
884

First retrieved item:
{
  "section_id": "42074235_UNLABELLED",
  "score": 172.0,
  "text": "Activating PIK3CA mutations occur in approximately 40% of hormone receptor-positive (HR+)/HER2-negative breast cancers and represent a major driver of endocrine resistance. The PI3Kα-selective inhibitor alpelisib, in combination with fulvestrant, significantly improves progression-free survival in patients with PIK3CA-mutant disease, as demonstrated in the SOLAR-1 trial. However, this therapeutic strategy is frequently complicated by treatment-induced hyperglycemia, a metabolic disturbance that promotes oxidative stress, mitochondrial dysfunction, and inflammatory signaling, thereby increasing cardiovascular vulnerability. Sodium-glucose cotransporter-2 (SGLT2) inhibitors have emerged as cardiometabolic modulators with benefits extending beyond glucose lowering. In this study, we used a human cardiomyocyte in vitro

In [7]:
# ============================================================
# CELL 7 — Retrieval ID Extraction
# ============================================================

def extract_node_id(item):
    """
    Extract the section/node identifier from one
    retrieval result.

    Adjust this function ONLY if the actual output
    schema uses a different field name.
    """

    if isinstance(item, str):
        return item

    if not isinstance(item, dict):
        return None

    candidate_keys = [
        "node_id",
        "section_id",
        "id",
        "node",
        "section"
    ]

    for key in candidate_keys:

        if key in item:

            value = item[key]

            if isinstance(value, str):
                return value

            if isinstance(value, dict):

                for nested_key in [
                    "node_id",
                    "section_id",
                    "id"
                ]:

                    if nested_key in value:
                        return value[nested_key]

    return None

In [8]:
# ============================================================
# CELL 8 — Verify ID Extraction
# ============================================================

example_items = example_run["retrieval"]

for i, item in enumerate(
    example_items[:5]
):

    print(
        i,
        "->",
        extract_node_id(item)
    )

0 -> 42074235_UNLABELLED
1 -> 42100630_EXPERIMENTAL APPROACH
2 -> 42014951_UNLABELLED
3 -> 41999978_RESULTS
4 -> 42091907_UNLABELLED


In [9]:
# ============================================================
# CELL 9 — Build Gold Evidence Sets
# ============================================================

gold_by_query = {}

for item in queries:

    query_id = item["id"]

    gold_ids = [
        evidence["node_id"]
        for evidence in item.get(
            "gold_evidence",
            []
        )
        if "node_id" in evidence
    ]

    gold_by_query[query_id] = set(
        gold_ids
    )

    print(
        query_id,
        "gold evidence:",
        len(gold_ids)
    )

Q1 gold evidence: 9
Q2 gold evidence: 5
Q3 gold evidence: 3
Q4 gold evidence: 7
Q5 gold evidence: 7
Q6 gold evidence: 5
Q7 gold evidence: 4
Q8 gold evidence: 10
Q9 gold evidence: 9
Q10 gold evidence: 10


In [10]:
# ============================================================
# CELL 10 — Metric Functions
# ============================================================

def precision_at_k(
    retrieved_ids,
    gold_ids,
    k
):
    retrieved = retrieved_ids[:k]

    if not retrieved:
        return 0.0

    hits = sum(
        1
        for node_id in retrieved
        if node_id in gold_ids
    )

    return hits / len(retrieved)


def recall_at_k(
    retrieved_ids,
    gold_ids,
    k
):
    if not gold_ids:
        return 0.0

    retrieved = retrieved_ids[:k]

    hits = sum(
        1
        for node_id in retrieved
        if node_id in gold_ids
    )

    return hits / len(gold_ids)


def f1_score(
    precision,
    recall
):
    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


def hit_rate_at_k(
    retrieved_ids,
    gold_ids,
    k
):
    retrieved = retrieved_ids[:k]

    return float(
        any(
            node_id in gold_ids
            for node_id in retrieved
        )
    )


def reciprocal_rank(
    retrieved_ids,
    gold_ids
):
    for rank, node_id in enumerate(
        retrieved_ids,
        start=1
    ):

        if node_id in gold_ids:
            return 1.0 / rank

    return 0.0


def average_precision(
    retrieved_ids,
    gold_ids
):
    if not gold_ids:
        return 0.0

    hits = 0
    precision_sum = 0.0

    for rank, node_id in enumerate(
        retrieved_ids,
        start=1
    ):

        if node_id in gold_ids:

            hits += 1

            precision_sum += (
                hits / rank
            )

    return (
        precision_sum /
        min(
            len(gold_ids),
            len(retrieved_ids)
        )
        if retrieved_ids
        else 0.0
    )

In [11]:
# ============================================================
# CELL 11 — Evaluation K Values
# ============================================================

K_VALUES = [
    1,
    3,
    5,
    10
]

print(
    "Evaluating at K =",
    K_VALUES
)

Evaluating at K = [1, 3, 5, 10]


In [12]:
# ============================================================
# CELL 12 — Per-Query Evaluation
# ============================================================

query_rows = []

for query_item in queries:

    query_id = query_item["id"]

    record = raw_runs[query_id]

    retrieval_items = (
        record["retrieval"]
    )

    retrieved_ids = []

    for item in retrieval_items:

        node_id = extract_node_id(
            item
        )

        if node_id is not None:
            retrieved_ids.append(
                node_id
            )

    gold_ids = gold_by_query[
        query_id
    ]

    row = {
        "query_id": query_id,
        "query": query_item["query"],
        "anchor_entity": query_item.get(
            "anchor_entity"
        ),
        "gold_count": len(gold_ids),
        "retrieved_count": len(
            retrieved_ids
        ),
        "latency_sec": record[
            "runtime"
        ]["latency_sec"]
    }

    for k in K_VALUES:

        p = precision_at_k(
            retrieved_ids,
            gold_ids,
            k
        )

        r = recall_at_k(
            retrieved_ids,
            gold_ids,
            k
        )

        row[
            f"precision@{k}"
        ] = p

        row[
            f"recall@{k}"
        ] = r

        row[
            f"f1@{k}"
        ] = f1_score(
            p,
            r
        )

        row[
            f"hit_rate@{k}"
        ] = hit_rate_at_k(
            retrieved_ids,
            gold_ids,
            k
        )

    row["MRR"] = reciprocal_rank(
        retrieved_ids,
        gold_ids
    )

    row["average_precision"] = (
        average_precision(
            retrieved_ids,
            gold_ids
        )
    )

    query_rows.append(row)


query_scores = pd.DataFrame(
    query_rows
)

query_scores

,query_id,query,anchor_entity,gold_count,retrieved_count,latency_sec,precision@1,recall@1,f1@1,hit_rate@1,...,precision@5,recall@5,f1@5,hit_rate@5,precision@10,recall@10,f1@10,hit_rate@10,MRR,average_precision
0,Q1,What are the cardiovascular risks of aromatase...,tamoxifen,9,884,0.447264,0.0,0.000000,0.000000,0.0,...,0.4,0.222222,0.285714,1.0,0.2,0.222222,0.210526,1.0,0.500000,0.191734
1,Q2,How are tumor-infiltrating lymphocytes (TILs) ...,tumor-infiltrating lymphocytes,5,947,0.910520,1.0,0.200000,0.333333,1.0,...,0.4,0.400000,0.400000,1.0,0.3,0.600000,0.400000,1.0,1.000000,0.491106
2,Q3,What are the advances and challenges in HER2-t...,trastuzumab,3,0,0.011846,0.0,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
3,Q4,What is the role of PIK3CA mutations in breast...,pik3ca,7,997,1.057349,0.0,0.000000,0.000000,0.0,...,0.2,0.142857,0.166667,1.0,0.1,0.142857,0.117647,1.0,0.333333,0.057618
4,Q5,What mechanisms of PD-L1 regulation and immuno...,pd-l1,7,666,0.124358,1.0,0.142857,0.250000,1.0,...,0.6,0.428571,0.500000,1.0,0.3,0.428571,0.352941,1.0,1.000000,0.586246
5,Q6,How is explainable AI being applied to identif...,explainable artificial intelligence,5,997,1.409812,0.0,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.012195,0.024178
6,Q7,What is known about olaparib and PARP inhibiti...,olaparib,4,947,1.653141,0.0,0.000000,0.000000,0.0,...,0.2,0.250000,0.222222,1.0,0.2,0.500000,0.285714,1.0,0.200000,0.154010
7,Q8,How is spatial transcriptomics being used to s...,spatial transcriptomics,10,966,1.939451,0.0,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.052632,0.080554
8,Q9,How is paclitaxel being evaluated and delivere...,paclitaxel,9,688,0.243092,0.0,0.000000,0.000000,0.0,...,0.4,0.222222,0.285714,1.0,0.2,0.222222,0.210526,1.0,0.333333,0.211338
9,Q10,What is the role of estrogen receptor status i...,estrogen receptor,10,519,0.081954,0.0,0.000000,0.000000,0.0,...,0.6,0.300000,0.400000,1.0,0.6,0.600000,0.600000,1.0,0.333333,0.530249


In [13]:
# ============================================================
# CELL 13 — Save Per-Query Scores
# ============================================================

QUERY_SCORE_PATH = (
    SCORES_DIR /
    "local_retrieval_scores_v1.csv"
)

query_scores.to_csv(
    QUERY_SCORE_PATH,
    index=False
)

print(
    "Saved:",
    QUERY_SCORE_PATH
)

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/scores/local_retrieval_scores_v1.csv


In [14]:
# ============================================================
# CELL 14 — Aggregate Metrics
# ============================================================

metric_columns = []

for k in K_VALUES:

    metric_columns.extend([
        f"precision@{k}",
        f"recall@{k}",
        f"f1@{k}",
        f"hit_rate@{k}"
    ])

metric_columns.extend([
    "MRR",
    "average_precision"
])

aggregate_rows = []

for metric in metric_columns:

    aggregate_rows.append({
        "metric": metric,
        "mean": query_scores[
            metric
        ].mean(),

        "median": query_scores[
            metric
        ].median(),

        "std": query_scores[
            metric
        ].std()
    })

aggregate_scores = pd.DataFrame(
    aggregate_rows
)

aggregate_scores

,metric,mean,median,std
0,precision@1,0.200000,0.000000,0.421637
1,recall@1,0.034286,0.000000,0.073525
2,f1@1,0.058333,0.000000,0.124536
3,hit_rate@1,0.200000,0.000000,0.421637
4,precision@3,0.266667,0.333333,0.262937
5,recall@3,0.115079,0.105556,0.135113
6,f1@3,0.158718,0.160256,0.175534
7,hit_rate@3,0.600000,1.000000,0.516398
8,precision@5,0.280000,0.300000,0.234758
9,recall@5,0.196587,0.222222,0.159472


In [15]:
# ============================================================
# CELL 15 — Save Aggregate Scores
# ============================================================

AGGREGATE_SCORE_PATH = (
    SCORES_DIR /
    "local_retrieval_aggregate_v1.csv"
)

aggregate_scores.to_csv(
    AGGREGATE_SCORE_PATH,
    index=False
)

print(
    "Saved:",
    AGGREGATE_SCORE_PATH
)

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/scores/local_retrieval_aggregate_v1.csv


In [16]:
# ============================================================
# CELL 16 — Key Metrics
# ============================================================

key_metrics = aggregate_scores[
    aggregate_scores["metric"].isin([
        "precision@1",
        "precision@3",
        "precision@5",
        "precision@10",
        "recall@1",
        "recall@3",
        "recall@5",
        "recall@10",
        "f1@5",
        "f1@10",
        "MRR",
        "average_precision"
    ])
].copy()

key_metrics

,metric,mean,median,std
0,precision@1,0.200000,0.000000,0.421637
1,recall@1,0.034286,0.000000,0.073525
4,precision@3,0.266667,0.333333,0.262937
5,recall@3,0.115079,0.105556,0.135113
8,precision@5,0.280000,0.300000,0.234758
9,recall@5,0.196587,0.222222,0.159472
10,f1@5,0.226032,0.253968,0.182429
12,precision@10,0.190000,0.200000,0.185293
13,recall@10,0.271587,0.222222,0.243869
14,f1@10,0.217736,0.210526,0.198493


In [17]:
# ============================================================
# CELL 17 — Query-Level Failure Analysis
# ============================================================

failure_columns = [
    "query_id",
    "anchor_entity",
    "gold_count",
    "retrieved_count",
    "precision@5",
    "recall@5",
    "f1@5",
    "hit_rate@5",
    "MRR",
    "latency_sec"
]

failure_table = query_scores[
    failure_columns
].sort_values(
    by="recall@5",
    ascending=True
)

failure_table

,query_id,anchor_entity,gold_count,retrieved_count,precision@5,recall@5,f1@5,hit_rate@5,MRR,latency_sec
2,Q3,trastuzumab,3,0,0.0,0.000000,0.000000,0.0,0.000000,0.011846
5,Q6,explainable artificial intelligence,5,997,0.0,0.000000,0.000000,0.0,0.012195,1.409812
7,Q8,spatial transcriptomics,10,966,0.0,0.000000,0.000000,0.0,0.052632,1.939451
3,Q4,pik3ca,7,997,0.2,0.142857,0.166667,1.0,0.333333,1.057349
0,Q1,tamoxifen,9,884,0.4,0.222222,0.285714,1.0,0.500000,0.447264
8,Q9,paclitaxel,9,688,0.4,0.222222,0.285714,1.0,0.333333,0.243092
6,Q7,olaparib,4,947,0.2,0.250000,0.222222,1.0,0.200000,1.653141
9,Q10,estrogen receptor,10,519,0.6,0.300000,0.400000,1.0,0.333333,0.081954
1,Q2,tumor-infiltrating lymphocytes,5,947,0.4,0.400000,0.400000,1.0,1.000000,0.910520
4,Q5,pd-l1,7,666,0.6,0.428571,0.500000,1.0,1.000000,0.124358


In [18]:
# ============================================================
# CELL 18 — Detailed Retrieval Diagnostics
# ============================================================

diagnostics = []

for query_item in queries:

    query_id = query_item["id"]

    record = raw_runs[query_id]

    retrieved_items = (
        record["retrieval"]
    )

    retrieved_ids = [
        extract_node_id(item)
        for item in retrieved_items
    ]

    retrieved_ids = [
        x for x in retrieved_ids
        if x is not None
    ]

    gold_ids = gold_by_query[
        query_id
    ]

    retrieved_set = set(
        retrieved_ids
    )

    missed = [
        node_id
        for node_id in gold_ids
        if node_id not in retrieved_set
    ]

    false_positives = [
        node_id
        for node_id in retrieved_ids
        if node_id not in gold_ids
    ]

    diagnostics.append({
        "query_id": query_id,

        "anchor_entity":
            query_item.get(
                "anchor_entity"
            ),

        "gold_count":
            len(gold_ids),

        "retrieved_count":
            len(retrieved_ids),

        "gold_retrieved":
            len(
                gold_ids &
                retrieved_set
            ),

        "missed_gold_count":
            len(missed),

        "missed_gold_ids":
            missed,

        "false_positive_count":
            len(false_positives),

        "false_positive_ids":
            false_positives,

        "retrieved_ids":
            retrieved_ids,

        "gold_ids":
            list(gold_ids)
    })


diagnostics_df = pd.DataFrame(
    diagnostics
)

diagnostics_df

,query_id,anchor_entity,gold_count,retrieved_count,gold_retrieved,missed_gold_count,missed_gold_ids,false_positive_count,false_positive_ids,retrieved_ids,gold_ids
0,Q1,tamoxifen,9,884,9,0,[],875,"[42074235_UNLABELLED, 42014951_UNLABELLED, 420...","[42074235_UNLABELLED, 42100630_EXPERIMENTAL AP...","[42100630_EXPERIMENTAL APPROACH, 41891505_RESU..."
1,Q2,tumor-infiltrating lymphocytes,5,947,5,0,[],942,"[42032133_UNLABELLED, 42018935_UNLABELLED, 420...","[42099744_UNLABELLED, 42032133_UNLABELLED, 420...","[42079614_INTRODUCTION, 42027950_UNLABELLED, 4..."
2,Q3,trastuzumab,3,0,0,3,"[42018935_UNLABELLED, 42041714_UNLABELLED, 420...",0,[],[],"[42018935_UNLABELLED, 42041714_UNLABELLED, 420..."
3,Q4,pik3ca,7,997,7,0,[],990,"[42032133_UNLABELLED, 42099744_UNLABELLED, 420...","[42032133_UNLABELLED, 42099744_UNLABELLED, 420...","[41968976_CONCLUSIONS, 41968987_RESULTS, 41968..."
4,Q5,pd-l1,7,666,7,0,[],659,"[42043855_UNLABELLED, 42013067_UNLABELLED, 418...","[42099744_UNLABELLED, 42032133_UNLABELLED, 420...","[41876831_UNLABELLED, 41939212_PURPOSE, 419303..."
5,Q6,explainable artificial intelligence,5,997,5,0,[],992,"[42099744_UNLABELLED, 42091742_UNLABELLED, 420...","[42099744_UNLABELLED, 42091742_UNLABELLED, 420...","[41884707_METHODS, 41968995_UNLABELLED, 421006..."
6,Q7,olaparib,4,947,4,0,[],943,"[42099744_UNLABELLED, 42074235_UNLABELLED, 419...","[42099744_UNLABELLED, 42074235_UNLABELLED, 419...","[42074514_RESULTS, 42035252_UNLABELLED, 419301..."
7,Q8,spatial transcriptomics,10,966,10,0,[],956,"[42032133_UNLABELLED, 42099744_UNLABELLED, 420...","[42032133_UNLABELLED, 42099744_UNLABELLED, 420...","[42032729_METHODS, 42088484_METHODS, 41870799_..."
8,Q9,paclitaxel,9,688,9,0,[],679,"[42032133_UNLABELLED, 42099744_UNLABELLED, 420...","[42032133_UNLABELLED, 42099744_UNLABELLED, 419...","[41991549_UNLABELLED, 41951086_UNLABELLED, 420..."
9,Q10,estrogen receptor,10,519,10,0,[],509,"[42099744_UNLABELLED, 41930306_DIAGNOSTIC PRED...","[42099744_UNLABELLED, 41930306_DIAGNOSTIC PRED...","[42100630_EXPERIMENTAL APPROACH, 42079614_INTR..."


In [19]:
# ============================================================
# CELL 19 — Save Retrieval Diagnostics
# ============================================================

DIAGNOSTICS_PATH = (
    SCORES_DIR /
    "local_retrieval_diagnostics_v1.json"
)

with open(
    DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        diagnostics,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    DIAGNOSTICS_PATH
)

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/scores/local_retrieval_diagnostics_v1.json


In [20]:
# ============================================================
# CELL 20 — Generate Summary Report
# ============================================================

report_path = (
    REPORTS_DIR /
    "local_retrieval_summary_v1.md"
)

def get_metric(metric_name):

    row = aggregate_scores[
        aggregate_scores["metric"]
        == metric_name
    ]

    if row.empty:
        return None

    return row.iloc[0]["mean"]


report = []

report.append(
    "# ARIA-Lite v2 — Local Retrieval Evaluation\n"
)

report.append(
    "## Benchmark\n"
)

report.append(
    f"- Benchmark: `{BENCHMARK_PATH.name}`\n"
    f"- Queries: {len(queries)}\n"
    f"- Retrieval run: `local_retrieval_v1`\n"
)

report.append(
    "\n## Aggregate Performance\n"
)

report.append(
    "| Metric | Mean |\n"
    "|---|---:|\n"
)

for metric in [
    "precision@1",
    "precision@3",
    "precision@5",
    "precision@10",
    "recall@1",
    "recall@3",
    "recall@5",
    "recall@10",
    "f1@5",
    "f1@10",
    "MRR",
    "average_precision"
]:

    value = get_metric(metric)

    if value is not None:

        report.append(
            f"| {metric} | {value:.4f} |\n"
        )

report.append(
    "\n## Query-Level Results\n"
)

report.append(
    "| Query | Anchor | "
    "Precision@5 | Recall@5 | F1@5 | MRR |\n"
    "|---|---|---:|---:|---:|---:|\n"
)

for _, row in query_scores.iterrows():

    report.append(
        f'| {row["query_id"]} | '
        f'{row["anchor_entity"]} | '
        f'{row["precision@5"]:.3f} | '
        f'{row["recall@5"]:.3f} | '
        f'{row["f1@5"]:.3f} | '
        f'{row["MRR"]:.3f} |\n'
    )

report.append(
    "\n## Notes\n"
)

report.append(
    "- Retrieval implementation was evaluated as-is.\n"
    "- No retrieval logic was modified during evaluation.\n"
    "- Gold evidence is taken from the benchmark's `gold_evidence` node IDs.\n"
    "- Metrics are based on retrieved section/node IDs.\n"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "".join(report)
    )

print(
    "Saved:",
    report_path
)

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/retrieval_only/retrieval_local/reports/local_retrieval_summary_v1.md


In [21]:
# ============================================================
# CELL 21 — Evaluation Outputs
# ============================================================

print("=" * 80)
print("LOCAL RETRIEVAL EVALUATION COMPLETE")
print("=" * 80)

print("\nScores:")

for path in sorted(
    SCORES_DIR.iterdir()
):

    print(
        " -",
        path.name
    )

print("\nReports:")

for path in sorted(
    REPORTS_DIR.iterdir()
):

    print(
        " -",
        path.name
    )

LOCAL RETRIEVAL EVALUATION COMPLETE

Scores:
 - local_retrieval_aggregate_v1.csv
 - local_retrieval_diagnostics_v1.json
 - local_retrieval_scores_v1.csv

Reports:
 - local_retrieval_summary_v1.md
